# Quiz 2：Plain CNN 與 VGG-19

無資料擴增。兩模型各三組 learning rate、五個 fold；依平均驗證 Top-1 選擇設定。

VGG-19：ImageNet 預訓練、11 類輸出、分類頭暖身後微調最後卷積區塊。

In [ ]:
from pathlib import Path
import sys
import subprocess
import json
from IPython.display import display, Image, Markdown
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'Quiz01_DatasetPreparation').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Please open this notebook inside the week3 directory.')
sys.path.insert(0, str(ROOT))
config = json.loads((ROOT / 'Quiz01_DatasetPreparation/Output/stage1_config.json').read_text(encoding='utf-8'))
device = 'cpu'
print('Project:', ROOT, '| Python:', sys.executable, '| device:', device)
display(config)

def run_script(script, *arguments):
    subprocess.run([sys.executable, str(ROOT / script), *map(str, arguments)], check=True, cwd=ROOT)

def show_file(path):
    path = ROOT / path
    if not path.exists():
        print('尚未產生：', path)
    elif path.suffix == '.csv':
        display(pd.read_csv(path))
    elif path.suffix == '.png':
        display(Image(filename=str(path)))
    elif path.suffix == '.md':
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        display(json.loads(path.read_text(encoding='utf-8')))


In [ ]:
display(pd.read_csv(ROOT / 'Quiz02_CNNClassification/Output/model_comparison.csv'))

## 超參數交叉驗證

設定 RUN_CV=True 會啟動 30 次基準訓練。先完成 Quiz 1；建議在 GPU 執行。

In [ ]:
RUN_CV = False
if RUN_CV:
    run_script('Quiz02_CNNClassification/Code/run.py', '--stage', 'cv', '--device', str(device))

In [ ]:
for name in ('plain', 'vgg19'):
    show_file(f'Quiz02_CNNClassification/Output/{name}/hyperparameters.csv')
    show_file(f'Quiz02_CNNClassification/Output/{name}/selection.json')

## 最終重訓與測試

建議先完成 Quiz 3 的 CV 才查看 test。設定 RUN_FINAL=True 才執行兩次重訓與測試。

In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    run_script('Quiz02_CNNClassification/Code/run.py', '--stage', 'final', '--device', str(device))
RUN_REPORT = False
if RUN_REPORT:
    run_script('report.py')

In [ ]:
show_file('Quiz02_CNNClassification/Output/model_comparison.csv')
for name in ('plain', 'vgg19'):
    show_file(f'Quiz02_CNNClassification/Output/{name}/test/metrics.json')
    show_file(f'Quiz02_CNNClassification/Output/{name}/test/roc_per_class.png')
    show_file(f'Quiz02_CNNClassification/Output/{name}/test/confusion_matrix.png')
show_file('Quiz02_CNNClassification/Output/report.md')